In [3]:
"""
Milestone 3 — Retrieval-Augmented Generation (RAG)
Rewritten with different implementation approaches; all outputs match the original.
"""
!pip install faiss-cpu #install FAISS
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

train = pd.read_csv('../data/train.csv')

In [4]:
# ── Setup ─────────────────────────────────────────────────────────────────
# Build the knowledge base with a list comprehension + row.get-style access
# instead of an explicit append loop.
print("Creating knowledge base")
kb = [str(row[row['answer']]) for _, row in train.iterrows()]

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)

# Use IndexFlatL2 constructed via faiss.index_factory instead of the direct
# class constructor (same index type/behavior).
embedding_dim = kb_embeddings.shape[1]
index = faiss.index_factory(embedding_dim, "Flat", faiss.METRIC_L2)
index.add(kb_embeddings)

print("Knowledge base successfully created")

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
ans_150 = str(row_150[row_150['answer']])

Creating knowledge base
Loading embedding model and creating index


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2605.00it/s]


Knowledge base successfully created


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 1753.06it/s]


In [5]:
# ── Question 1 ────────────────────────────────────────────────────────────
# Zip labels/scores into pairs and build the map with a dict comprehension
# instead of dict(zip(...)).
result = zs(prompt_150, candidate_labels=labels_150)
score_map = {label: score for label, score in zip(result["labels"], result["scores"])}
ground_truth_score = score_map[ans_150]
print(round(ground_truth_score, 3))
# 0.384

0.384


In [6]:
# ── Question 2 ────────────────────────────────────────────────────────────
# Find the rank using list.index() on a converted list instead of
# np.where(), with a plain try/except for "not found".
query_emb = model.encode([prompt_150], show_progress_bar=False)
distances, top10_indices = index.search(query_emb, 10)
top10_list = top10_indices[0].tolist()

target_idx = 150
try:
    rank = top10_list.index(target_idx) + 1
except ValueError:
    rank = None
print(rank)
# 10

10


In [7]:
# ── Question 3 ────────────────────────────────────────────────────────────
# Pair up (doc_index, score) tuples and sort with sorted(..., reverse=True)
# instead of np.argsort on a separate scores array.
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in top10_list]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

scored_pairs = list(zip(top10_list, ce_scores))
scored_pairs.sort(key=lambda pair: pair[1], reverse=True)
sorted_indices = [doc_idx for doc_idx, _ in scored_pairs]

try:
    ce_rank = sorted_indices.index(target_idx) + 1
except ValueError:
    ce_rank = None
print(ce_rank)
# 1

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2581.13it/s]


1


In [8]:
# ── Question 4 ────────────────────────────────────────────────────────────
# Build the context string with str.join over a generator, and count tokens
# via the tokenizer's __call__ length using len(encoding['input_ids'])
# accessed through .get on the BatchEncoding instead of bracket indexing.
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

query_emb_42 = model.encode([prompt_42], show_progress_bar=False)
_, indices_42 = index.search(query_emb_42, 5)
docs_5 = (kb[i] for i in indices_42[0])  # generator instead of list comp

context = " ".join(docs_5)
text = "Context: {} Question: {}".format(context, prompt_42)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
encoding = tokenizer(text, truncation=False)
num_tokens = len(encoding.input_ids)
print(num_tokens)
# 216

216


In [9]:
# ── Question 5 ────────────────────────────────────────────────────────────
# Fetch the true document via kb[target_idx] (reusing target_idx=150)
# and build the RAG string with an f-string wrapped in a small helper
# function instead of an inline literal.
def build_rag_string(context_doc: str, prompt: str) -> str:
    return f"Context: {context_doc} Question: {prompt}"

true_document = kb[target_idx]
rag_text = build_rag_string(true_document, prompt_150)

rag_result = zs(rag_text, candidate_labels=labels_150)
rag_score_map = {label: score for label, score in zip(rag_result["labels"], rag_result["scores"])}
rag_ground_truth_score = rag_score_map[ans_150]
print(round(rag_ground_truth_score, 3))
# 0.989

0.989


In [10]:
# ── Question 6 ────────────────────────────────────────────────────────────
# Reuse the build_rag_string helper for the adversarial context too.
adv_doc = kb[999]
adv_rag_text = build_rag_string(adv_doc, prompt_150)

adv_result = zs(adv_rag_text, candidate_labels=labels_150)
adv_score_map = {label: score for label, score in zip(adv_result["labels"], adv_result["scores"])}
adv_ground_truth_score = adv_score_map[ans_150]
print(round(adv_ground_truth_score, 3))
# 0.529


0.529


In [11]:
# ── Question 7 ────────────────────────────────────────────────────────────
# Use any() with a generator expression instead of a manual for/break loop
# to detect a hit.
def is_hit(correct_answer: str, doc_indices) -> bool:
    return any(correct_answer in kb[doc_idx] for doc_idx in doc_indices)

hit_flags = []
for idx in range(100):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_answer = str(row[row['answer']])

    query_emb = model.encode([prompt], show_progress_bar=False)
    _, indices_top5 = index.search(query_emb, 5)

    hit_flags.append(is_hit(correct_answer, indices_top5[0]))

hit_rate = (sum(hit_flags) / len(hit_flags)) * 100
print(round(hit_rate, 1))
# 73.0


73.0


In [12]:
# ── Question 8 ────────────────────────────────────────────────────────────
# Full retrieve → rerank → augment → predict → score pipeline, restructured
# into small helper functions instead of one long inline loop body.
OPTION_LETTERS = ['A', 'B', 'C', 'D', 'E']

def retrieve_top5(prompt: str):
    query_emb = model.encode([prompt], show_progress_bar=False)
    _, top5_idx = index.search(query_emb, 5)
    idxs = top5_idx[0]
    return idxs, [kb[j] for j in idxs]

def rerank_best(prompt: str, docs: list) -> str:
    pairs = [[prompt, doc] for doc in docs]
    scores = cross_encoder.predict(pairs)
    return docs[int(np.argmax(scores))]

def predict_ranked_letters(rag_text: str, row) -> list:
    text_to_letter = {str(row[letter]): letter for letter in OPTION_LETTERS}
    candidate_labels = [str(row[letter]) for letter in OPTION_LETTERS]
    zs_res = zs(rag_text, candidate_labels=candidate_labels)
    ranked = sorted(zip(zs_res["labels"], zs_res["scores"]), key=lambda p: p[1], reverse=True)
    return [text_to_letter.get(label) for label, _ in ranked]

def average_precision_at_3(true_letter: str, ranked_letters: list) -> float:
    top3 = ranked_letters[:3]
    return 1.0 / (top3.index(true_letter) + 1) if true_letter in top3 else 0.0

map3_scores = []
for i in range(20):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    true_letter = row['answer']

    _, docs_5 = retrieve_top5(prompt)
    best_doc = rerank_best(prompt, docs_5)
    rag_text_local = build_rag_string(best_doc, prompt)

    top3_letters = predict_ranked_letters(rag_text_local, row)[:3]
    map3_scores.append(average_precision_at_3(true_letter, top3_letters))

avg_map3 = float(np.mean(map3_scores))
print(round(avg_map3, 3))
# 0.975

0.975
